<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_7_Harbinger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏢 RetailMax Employee Directory Analysis Scenario (PySpark)

### **Scenario Description**
The HR department of RetailMax exported their employees' directory as a comma-separated values file (`employees.csv`). However, the export was generated **without header column names**.

In this notebook, we will demonstrate:
1. Setting up Apache Spark in our environment.
2. Loading data with a missing header (and seeing how PySpark treats the first row as columns when `header=True` is set).
3. Filtering, selecting, sorting, and aggregating records using the inferred column names.
4. **Best Practice**: How to fix the missing header issue by applying a custom schema.

---

## 🛠️ Step 1: Install Apache PySpark
We install the library to set up Spark on our local run environment.

In [ ]:
# Install PySpark client package
!pip install pyspark

## 🚀 Step 2: Import SparkSession
Import the necessary class from `pyspark.sql` to initialize our environment.

In [ ]:
from pyspark.sql import SparkSession

## ⚙️ Step 3: Initialize SparkSession
We create a Spark session with the app name `"RetailMax"`.

In [ ]:
# Initialize SparkSession builder
spark = SparkSession.builder \
    .appName("RetailMax") \
    .getOrCreate()

## 📂 Step 4: Load Employees CSV Data
> ⚠️ **Warning**: The CSV file lacks headers. Setting `header=True` forces PySpark to treat the first row (`1001, Rahul Sharma, Finance, 65000`) as the schema header. Let's see how this affects our column names.

In [ ]:
# Load CSV file. Inferred columns: '1001', 'Rahul Sharma', 'Finance', '65000'
df = spark.read.csv(
    "employees.csv",
    header=True,
    inferSchema=True
)

### Display the DataFrame
Note the headers are named after the first row of data.

In [ ]:
df.show()

### Record Count
Count the total records in the DataFrame (note: 1 row is lost as the header!).

In [ ]:
df.count()

### Print Schema
Look at the data types PySpark inferred for the columns.

In [ ]:
df.printSchema()

### Columns List
Verify column list names.

In [ ]:
df.columns

### Summary Statistics
Get standard statistics (mean, stddev, min, max) for columns.

In [ ]:
df.describe().show()

## 🔍 Step 5: Filter Employees by Department
We filter the records where the department column (named `'Finance'` due to inferred headers) equals `'Finance'`.

In [ ]:
# Filter rows matching department 'Finance'
df.filter(
    df.Finance == "Finance"
).show()

## 📋 Step 6: Select Name and Salary Columns
We select the name column (named `'Rahul Sharma'`) and salary column (named `'65000'`).

In [ ]:
# Select and display employee names and salaries
df.select(
    "Rahul Sharma",
    "65000"
).show()

## 📈 Step 7: Order Employees by Salary
Sort the DataFrame by salary (column `'65000'`) in descending order.

In [ ]:
# Sort salaries descending
df.orderBy(
    "65000",
    ascending=False
).show()

## 📊 Step 8: Group Employees by Department
Group records by department (column `'Finance'`) and count the employees in each group.

In [ ]:
# Group and count departments
df.groupBy(
    "Finance"
).count().show()

## 💰 Step 9: Calculate Average Salary per Department
We import `avg` aggregation function and compute the average salary (column `'65000'`) grouped by department (column `'Finance'`).

In [ ]:
from pyspark.sql.functions import avg

# Compute mean salary by department
df.groupBy(
    "Finance"
).agg(
    avg("65000")
).show()

---
## 💡 Step 10: How to Fix the Missing Header (Best Practice)
To avoid columns named after data records (`Rahul Sharma`, `65000`, etc.), we should define an explicit schema manually using `StructType` and `StructField` and read without setting `header=True`.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define the schema matching employees.csv format
schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("employee_name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True)
])

# Read data with manual schema
df_fixed = spark.read.csv(
    "employees.csv",
    header=False,
    schema=schema
)

df_fixed.show(5)